# 01 — Attack Analysis

Exploratory look at the generated DriftGuard benchmark: class balance, attack-family breakdown, and qualitative examples of MINJA-style vs. AgentPoison-style injections vs. benign drift vs. ordinary learning.

Run `scripts/generate_benchmark.py` first to produce `data/benchmark.jsonl`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from driftguard.benchmark.labelling import load_benchmark, benchmark_stats

BENCHMARK_PATH = "../data/benchmark.jsonl"
entries = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(entries)} benchmark entries")

In [ ]:
stats = benchmark_stats(entries)
stats

In [ ]:
df = pd.DataFrame([
    {
        "label": e.label,
        "attack_family": e.attack_family,
        "content_len": len(e.content),
        "trust_level": e.provenance["trust_level"],
        "origin": e.provenance["origin"],
    }
    for e in entries
])
df.head()

## Class balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["label"].value_counts().plot(kind="bar", ax=ax, color=["#e63946", "#2a9d8f", "#457b9d"])
ax.set_title("Benchmark class balance")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## Attack family breakdown (malicious only)

In [ ]:
mal = df[df["label"] == "malicious"]
fig, ax = plt.subplots(figsize=(7, 4))
mal["attack_family"].value_counts().plot(kind="barh", ax=ax, color="#e63946")
ax.set_title("Malicious entries by attack family")
plt.tight_layout()
plt.show()

## Trust level distribution by class

Provenance trust level alone should *not* cleanly separate malicious from benign — that is the whole point of DriftGuard needing the other three feature families.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="label", y="trust_level", ax=ax)
ax.set_title("Provenance trust_level by class")
plt.tight_layout()
plt.show()

## Content length distribution by class

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x="content_len", hue="label", element="step", stat="density", common_norm=False, ax=ax)
ax.set_title("Write content length by class")
plt.tight_layout()
plt.show()

## Qualitative examples

One sample per attack family / class, to sanity-check the generators produce realistic-looking text.

In [ ]:
for family in df["attack_family"].unique():
    example = next(e for e in entries if e.attack_family == family)
    print(f"[{example.label} / {family}]")
    print(f"  query:   {example.last_query}")
    print(f"  content: {example.content}")
    print()